# Phase 1 — PP-OCRv5_server_det theo đúng pipeline đánh giá CHÍNH THỨC của PaddleOCR

Notebook này chạy **đúng script `tools/eval.py` của repo PaddleOCR** (tag `v3.3.3`), với:

| Thành phần | Dùng gì | Nguồn |
|---|---|---|
| Code đánh giá | `tools/eval.py` + `ppocr` của repo, không sửa | GitHub PaddlePaddle/PaddleOCR, tag v3.3.3 |
| Config | `configs/det/PP-OCRv5/PP-OCRv5_server_det.yml`, không sửa tham số nào | repo |
| Weights | `PP-OCRv5_server_det_pretrained.pdparams` (training checkpoint chính thức) | link chính thức trên docs PaddleOCR |
| Tiền xử lý test | `DetResizeForTest` mặc định của config = **cạnh ngắn ≥ 736** | config |
| Hậu xử lý | `DBPostProcess` thresh 0.3, box_thresh 0.6, unclip 1.5 | config |
| Thước đo | `DetMetric` → `DetectionIoUEvaluator` (IoU > 0,5) | repo |
| Tập đánh giá | ICDAR2015 test (500 ảnh) + nhãn chính thức `test_icdar2015_label.txt` — **đây là tập Eval mặc định trong chính config của repo** | ảnh: Kaggle `bestofbests9/icdar2015`; nhãn: link chính thức PaddleOCR |

**Phần KHÔNG thể giống paper:** Hmean 83,8 được công bố trên một "general evaluation set" nội bộ, không mô tả và không công khai. Hai tech report (arXiv 2507.05595, 2603.24373) không có bảng detection nào. Vì vậy 83,8 vẫn là **NOT VERIFIED**; notebook này chỉ khẳng định: *pipeline, config, weights và thước đo giống hệt bản chính thức*.

### Cài đặt trên Kaggle
- **Accelerator: GPU T4** · **Internet: On**
- **Add Input → Datasets**: https://www.kaggle.com/datasets/bestofbests9/icdar2015

## 0. Config

In [ ]:
import os, sys, re, json, time, hashlib, platform, subprocess, shutil, datetime, importlib, importlib.util
import urllib.request, urllib.error, socket
from pathlib import Path
from collections import Counter

PADDLE_PKG = 'paddlepaddle-gpu==3.3.1'          # Kaggle Python 3.13: wheel cp313 chỉ có từ 3.2.0
PADDLE_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
PADDLEOCR_REPO = 'https://github.com/PaddlePaddle/PaddleOCR.git'
PADDLEOCR_TAG = 'v3.3.3'
PADDLEOCR_COMMIT = 'c0522b89e2218741f174ec8b45588dec93fa66b3'   # commit của tag v3.3.3
CONFIG_REL = 'configs/det/PP-OCRv5/PP-OCRv5_server_det.yml'
WEIGHTS_URL = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_det_pretrained.pdparams'
LABEL_URL = 'https://paddleocr.bj.bcebos.com/dataset/test_icdar2015_label.txt'
EXPECTED_LABEL_SHA256 = '07a7dafd27fe0b4e10ba37568ccaa524b716efca3b123ebc4f7cf87e418aeb96'   # tải ngày 2026-10-31

# Số tham chiếu (chỉ để so sánh, không dùng để chỉnh gì)
PUBLISHED = {
    'docs_text_detection_module': {'hmean': 0.838, 'eval_set': 'general evaluation set (không mô tả, không công khai)',
                                   'url': 'https://www.paddleocr.ai/main/en/version3.x/module_usage/text_detection.html'},
    'docs_ppocrv5_intro_avg': {'hmean': 0.827, 'eval_set': 'internal multi-scenario set, 12 nhóm (không công khai)',
                               'url': 'https://www.paddleocr.ai/latest/en/version3.x/algorithm/PP-OCRv5/PP-OCRv5.html'},
    'docs_ppocrv5_intro_general_scenario': {'hmean': 0.797, 'eval_set': 'nhóm General Scenario của bộ nội bộ trên'},
}
PREV_MODULE_RUN = {'precision': 0.5833333333333334, 'recall': 0.42465093885411653, 'hmean': 0.49150181108943997,
                   'note': 'lần chạy 2026-10-30 qua paddleocr.TextDetection (resize_long 960, inference model)'}

# Dùng chung với các cell copy từ notebook Phase 1
ICDAR15_TEST_IMG_DIR = None
ICDAR15_TEST_GT_DIR = None
ICDAR15_PADDLE_LABEL = None
ALLOW_RRC_DOWNLOAD = False
EXPECTED_N_IMAGES = 500
IOU_THR = 0.5
IGNORE_AREA_PRECISION_THR = 0.5
IGNORE_TAGS = ('###', '*')

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ART = WORK / 'artifacts' / 'phase1_paddle_official'
REPO = WORK / 'PaddleOCR'
OFFICIAL = WORK / 'official_files'
DATA_DIR = WORK / 'train_data' / 'icdar2015' / 'text_localization'
for d in (ART, OFFICIAL):
    d.mkdir(parents=True, exist_ok=True)
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat()
INSTALL_LOG = ART / 'pip_install.log'


def sh(cmd, log=None, check=True, cwd=None, env=None):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, cwd=cwd, env=env)
    out = p.stdout + p.stderr
    if log is not None:
        with open(log, 'a', encoding='utf-8') as f:
            f.write(f'$ {cmd}\n{out}\n')
    if check and p.returncode != 0:
        print(out[-3000:])
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')
    return out


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')


print('Artifacts ->', ART)

## 1. Preflight: GPU, mạng, dữ liệu (dừng sớm trước bước cài lâu)

In [ ]:
print(sh('nvidia-smi', check=False))
print('python', sys.version)

NET_URLS = {
    'pypi.org': 'https://pypi.org/simple/pip/',
    'www.paddlepaddle.org.cn': PADDLE_INDEX,
    'github.com': 'https://github.com',
    'paddle-model-ecology.bj.bcebos.com': 'https://paddle-model-ecology.bj.bcebos.com',
    'paddleocr.bj.bcebos.com': 'https://paddleocr.bj.bcebos.com',
}
net = {}
for host, url in NET_URLS.items():
    r = {'dns': False, 'https': False, 'error': None}
    try:
        socket.getaddrinfo(host, 443)
        r['dns'] = True
        try:
            urllib.request.urlopen(url, timeout=20).close()
            r['https'] = True
        except urllib.error.HTTPError:
            r['https'] = True
    except Exception as e:
        r['error'] = f'{type(e).__name__}: {e}'
    net[host] = r
    print(f'{host:38s} dns={r["dns"]!s:5s} https={r["https"]!s:5s} {r["error"] or ""}')
write_json(ART / 'network_check.json', net)
bad = [h for h, r in net.items() if not r['https']]
if bad:
    raise RuntimeError(f'Không tới được {bad}. Bật Settings → Internet: On (tài khoản phải xác minh SĐT).')

In [ ]:
# Kiểm tra nhanh dữ liệu TRƯỚC khi cài (bước cài mất ~12 phút) — chỉ dùng thư viện chuẩn
ICDAR15_DATASET_URL = 'https://www.kaggle.com/datasets/bestofbests9/icdar2015'


def quick_data_check(root=INPUT_ROOT, n=EXPECTED_N_IMAGES):
    want = set(range(1, n + 1))
    img_hits, gt_hits, label_hits, nb_outputs = [], [], [], []
    for dp, dn, fn in os.walk(root):
        ids_img = {int(m.group(1)) for f in fn if (m := re.fullmatch(r'img_(\d+)\.(?:jpg|jpeg|png)', f, re.I))}
        ids_gt = {int(m.group(1)) for f in fn if (m := re.fullmatch(r'gt_img_(\d+)\.txt', f))}
        if ids_img == want:
            img_hits.append(dp)
        if ids_gt == want:
            gt_hits.append(dp)
        label_hits += [os.path.join(dp, f) for f in fn if re.fullmatch(r'.*test.*label.*\.txt', f, re.I)]
        if '__notebook__.ipynb' in fn:
            nb_outputs.append(dp)
    return img_hits, gt_hits, label_hits, nb_outputs


if ICDAR15_TEST_IMG_DIR or ALLOW_RRC_DOWNLOAD:
    print('Bỏ qua quick check (đã đặt đường dẫn tay hoặc bật ALLOW_RRC_DOWNLOAD)')
else:
    img_hits, gt_hits, label_hits, nb_outputs = quick_data_check()
    print('ảnh test (img_1..500):', img_hits)
    print('GT test (gt_img_1..500):', gt_hits, '| label PaddleOCR:', label_hits)
    if not img_hits or not (gt_hits or label_hits or ICDAR15_TEST_GT_DIR or ICDAR15_PADDLE_LABEL):
        hint = ''
        if nb_outputs:
            hint = (f'\nĐang gắn một NOTEBOOK ({nb_outputs[0]}): Kaggle chỉ mount output của notebook đó, KHÔNG kèm dữ liệu. '
                    f'Gỡ nó ra, rồi Add Input → Datasets → {ICDAR15_DATASET_URL}')
        raise RuntimeError(f'Chưa có ICDAR2015 test trong {INPUT_ROOT}.{hint}\n'
                           f'Gợi ý dataset: {ICDAR15_DATASET_URL} (Version 3, 136.35 MB, thư mục Detection/ch4_test_images).')
    print('quick data check OK')

## 2. Cài PaddlePaddle + clone repo PaddleOCR đúng tag

In [ ]:
FREEZE_BEFORE = sh([sys.executable, '-m', 'pip', 'freeze'], check=False)
(ART / 'pip_freeze_before_install.txt').write_text(FREEZE_BEFORE, encoding='utf-8')
sh([sys.executable, '-m', 'pip', 'install', '-q', PADDLE_PKG, '-i', PADDLE_INDEX], log=INSTALL_LOG)
PADDLE_SOURCE = f'pip install {PADDLE_PKG} -i {PADDLE_INDEX}'

if not REPO.exists():
    sh(['git', 'clone', '--depth', '1', '--branch', PADDLEOCR_TAG, PADDLEOCR_REPO, str(REPO)], log=INSTALL_LOG)
REPO_COMMIT = sh(['git', 'rev-parse', 'HEAD'], cwd=REPO).strip()
print('PaddleOCR HEAD:', REPO_COMMIT)
assert REPO_COMMIT == PADDLEOCR_COMMIT, f'commit {REPO_COMMIT} khác commit của tag {PADDLEOCR_TAG} ({PADDLEOCR_COMMIT})'
sh([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], log=INSTALL_LOG)
print('install OK — log:', INSTALL_LOG)

In [ ]:
# paddlepaddle-gpu kéo theo các gói nvidia-* (cu12.6) và có thể hạ cấp NCCL mà torch có sẵn của Kaggle cần.
# `import paddleocr` -> paddlex -> modelscope -> import torch -> lỗi `undefined symbol: ncclCommShrink`.
# Pipeline Paddle không dùng torch: torch hỏng thì gỡ khỏi phiên Kaggle này và ghi lại (không đụng tới Paddle).
import importlib, importlib.util


def _pkg_lines(freeze):
    return {ln.split('==')[0].lower(): ln for ln in freeze.splitlines()
            if '==' in ln and ln.lower().startswith(('nvidia-', 'torch'))}


freeze_after = sh([sys.executable, '-m', 'pip', 'freeze'], check=False)
before, after = _pkg_lines(FREEZE_BEFORE), _pkg_lines(freeze_after)
changes = [f'{k}: {before.get(k, "-")} -> {after.get(k, "-")}' for k in sorted(set(before) | set(after))
           if before.get(k) != after.get(k)]
(ART / 'nvidia_torch_pkg_changes.txt').write_text('\n'.join(changes) + '\n', encoding='utf-8')
print('nvidia-*/torch thay đổi khi cài Paddle:', *changes, sep='\n  ')

TORCH_ACTION = 'torch không có sẵn'
if importlib.util.find_spec('torch') is not None:
    p = subprocess.run([sys.executable, '-c', 'import torch'], capture_output=True, text=True)
    if p.returncode == 0:
        TORCH_ACTION = 'torch vẫn import được — giữ nguyên'
    else:
        err = (p.stderr.strip().splitlines() or ['?'])[-1]
        sh([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], log=INSTALL_LOG)
        importlib.invalidate_caches()
        assert importlib.util.find_spec('torch') is None, 'gỡ torch chưa xong'
        TORCH_ACTION = (f'torch hỏng sau khi cài Paddle ({err}) -> đã gỡ torch/torchvision/torchaudio '
                        f'khỏi phiên Kaggle này; pipeline Paddle không dùng torch')
print(TORCH_ACTION)

In [ ]:
import paddle
print('paddle', paddle.__version__, '| commit', paddle.version.commit)
print('CUDA:', paddle.is_compiled_with_cuda(), '| cuda', paddle.version.cuda(), '| cudnn', paddle.version.cudnn())
assert paddle.is_compiled_with_cuda(), 'Paddle không có CUDA'
paddle.utils.run_check()

# ppocr import rất nhiều module (albumentations, skimage, lmdb...) ngay khi khởi động: thử trước để báo lỗi rõ
probe = subprocess.run([sys.executable, '-c',
                        'import tools.program, ppocr.data, ppocr.modeling.architectures, ppocr.postprocess, ppocr.metrics; print("PPOCR_IMPORT_OK")'],
                       capture_output=True, text=True, cwd=REPO)
if 'PPOCR_IMPORT_OK' not in probe.stdout:
    print(probe.stderr[-3000:])
    raise RuntimeError('import ppocr (repo) lỗi — xem traceback ở trên')
print('import ppocr OK')

In [ ]:
# env_versions.txt + git_commit.txt
import importlib.metadata as md


def pkg_ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


env = {
    'utc_time': RUN_STARTED, 'python': sys.version, 'platform': platform.platform(),
    'paddlepaddle': paddle.__version__, 'paddle_commit': paddle.version.commit,
    'paddle_cuda': paddle.version.cuda(), 'paddle_cudnn': paddle.version.cudnn(),
    'paddleocr_repo_tag': PADDLEOCR_TAG, 'paddleocr_repo_commit': REPO_COMMIT,
    **{k: pkg_ver(k) for k in ('numpy', 'opencv-python', 'opencv-contrib-python', 'shapely', 'scikit-image',
                               'albumentations', 'albucore', 'pyclipper', 'lmdb')},
    'gpu': sh('nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader', check=False).strip(),
    'install_cmds': [PADDLE_SOURCE, f'git clone --branch {PADDLEOCR_TAG} {PADDLEOCR_REPO}', 'pip install -r requirements.txt'],
    'torch_action': TORCH_ACTION,
}
freeze = sh([sys.executable, '-m', 'pip', 'freeze'], check=False)
(ART / 'env_versions.txt').write_text('\n'.join(f'{k}: {v}' for k, v in env.items()) + '\n\n# pip freeze\n' + freeze,
                                      encoding='utf-8')
(ART / 'git_commit.txt').write_text(
    f'PaddleOCR repo  tag={PADDLEOCR_TAG}  HEAD={REPO_COMMIT}  (clone trực tiếp, không sửa file nào)\n'
    f'Paddle          pip={paddle.__version__}  build_commit={paddle.version.commit}\n', encoding='utf-8')
dirty = sh(['git', 'status', '--porcelain'], cwd=REPO).strip()
assert not dirty, f'repo PaddleOCR bị sửa: {dirty}'
print(json.dumps(env, indent=2, default=str))

## 3. Tải weights + nhãn chính thức, audit nhãn với GT gốc của ICDAR2015

In [ ]:
def download(url, dst):
    if not dst.exists():
        urllib.request.urlretrieve(url, dst)
    return {'url': url, 'path': str(dst), 'bytes': dst.stat().st_size, 'sha256': sha256_file(dst)}


WEIGHTS = OFFICIAL / Path(WEIGHTS_URL).name
LABEL = OFFICIAL / Path(LABEL_URL).name
downloads = {'weights': download(WEIGHTS_URL, WEIGHTS), 'label': download(LABEL_URL, LABEL)}
write_json(ART / 'official_downloads.json', downloads)
print(json.dumps(downloads, indent=2))
if downloads['label']['sha256'] != EXPECTED_LABEL_SHA256:
    print('WARN: file nhãn chính thức khác bản đã kiểm tra ngày 2026-10-31 — xem label audit bên dưới')
(ART / 'checkpoint_sha256.txt').write_text(
    f'{downloads["weights"]["sha256"]}  {downloads["weights"]["bytes"]:>12}  {WEIGHTS.name}\nsource: {WEIGHTS_URL}\n',
    encoding='utf-8')

In [ ]:
# == CORE ==
import numpy as np
from shapely.geometry import Polygon

ID_RE_GT = re.compile(r'gt_img_(\d+)\.txt')
ID_RE_IMG = re.compile(r'img_(\d+)\.(?:jpg|jpeg|png)', re.I)
LABEL_RE = re.compile(r'.*test.*label.*\.txt', re.I)


def scan_dirs(roots):
    gt_dirs, img_dirs, label_files = {}, {}, []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for dp, dn, fn in os.walk(root):
            g = {int(m.group(1)) for f in fn if (m := ID_RE_GT.fullmatch(f))}
            i = {int(m.group(1)) for f in fn if (m := ID_RE_IMG.fullmatch(f))}
            if g:
                gt_dirs[Path(dp)] = g
            if i:
                img_dirs[Path(dp)] = i
            label_files += [Path(dp) / f for f in fn if LABEL_RE.fullmatch(f)]
    return gt_dirs, img_dirs, label_files


def pick_one(cands, what):
    if len(cands) == 1:
        return cands[0]
    test_like = [c for c in cands if 'test' in str(c).lower()]
    if len(test_like) == 1:
        return test_like[0]
    raise RuntimeError(f'{what}: {len(cands)} ứng viên, hãy điền tay trong CONFIG: {[str(c) for c in cands]}')


def resolve_icdar15(roots, n_expected=EXPECTED_N_IMAGES):
    want = set(range(1, n_expected + 1))
    gt_dirs, img_dirs, label_files = scan_dirs(roots)
    if ICDAR15_TEST_IMG_DIR:
        img_dir = Path(ICDAR15_TEST_IMG_DIR)
    else:
        cands = [d for d, ids in img_dirs.items() if ids == want]
        if not cands:
            raise RuntimeError(f'Không tìm thấy thư mục chứa đúng img_1..img_{n_expected}. Các thư mục ảnh thấy được: '
                               f'{[(str(d), len(ids)) for d, ids in img_dirs.items()][:10]}')
        img_dir = pick_one(cands, 'ICDAR2015 test image dir')
    if ICDAR15_PADDLE_LABEL:
        return img_dir, Path(ICDAR15_PADDLE_LABEL), 'paddle_label'
    if ICDAR15_TEST_GT_DIR:
        return img_dir, Path(ICDAR15_TEST_GT_DIR), 'rrc'
    gt_cands = [d for d, ids in gt_dirs.items() if ids == want]
    if gt_cands:
        return img_dir, pick_one(gt_cands, 'ICDAR2015 test GT dir'), 'rrc'
    if label_files:
        return img_dir, pick_one(label_files, 'PaddleOCR label file'), 'paddle_label'
    raise RuntimeError('Không tìm thấy GT test (gt_img_*.txt hoặc *test*label*.txt). Điền tay trong CONFIG.')


def parse_rrc_gt_file(path):
    # dòng RRC: x1,y1,...,x4,y4,transcription (transcription có thể chứa dấu phẩy)
    items = []
    with open(path, encoding='utf-8-sig') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip().lstrip(chr(0xFEFF))
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 9:
                raise ValueError(f'{path}:{ln}: cần 8 toạ độ + text, nhận {line!r}')
            pts = np.array([float(x) for x in parts[:8]], dtype=np.float64).reshape(4, 2)
            text = ','.join(parts[8:])
            items.append({'points': pts, 'text': text, 'ignore': text in IGNORE_TAGS})
    return items


def parse_paddle_label(path):
    gts = {}
    with open(path, encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            rel, js = line.rstrip('\n').split('\t', 1)
            m = ID_RE_IMG.fullmatch(Path(rel).name)
            if not m:
                raise ValueError(f'{path}:{ln}: tên ảnh lạ {rel!r}')
            gts[int(m.group(1))] = [
                {'points': np.asarray(a['points'], dtype=np.float64).reshape(-1, 2),
                 'text': a['transcription'], 'ignore': a['transcription'] in IGNORE_TAGS}
                for a in json.loads(js)]
    return gts


def load_icdar15(img_dir, gt_src, fmt, n_expected=EXPECTED_N_IMAGES):
    img_dir, gt_src = Path(img_dir), Path(gt_src)
    imgs = {int(m.group(1)): img_dir / f for f in os.listdir(img_dir) if (m := ID_RE_IMG.fullmatch(f))}
    if fmt == 'rrc':
        gts = {int(m.group(1)): parse_rrc_gt_file(gt_src / f)
               for f in os.listdir(gt_src) if (m := ID_RE_GT.fullmatch(f))}
    else:
        gts = parse_paddle_label(gt_src)
    if set(imgs) != set(gts):
        raise ValueError(f'Lệch id ảnh/GT: chỉ có ảnh={sorted(set(imgs) - set(gts))[:10]}, '
                         f'chỉ có GT={sorted(set(gts) - set(imgs))[:10]}')
    if len(imgs) != n_expected:
        raise ValueError(f'Có {len(imgs)} ảnh, kỳ vọng {n_expected} (ICDAR2015 test)')
    return [{'image_id': f'img_{k}', 'path': str(imgs[k]), 'gt': gts[k]} for k in sorted(imgs)]


def _valid_poly(pts):
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 3 or not np.all(np.isfinite(pts)):
        return None
    p = Polygon(pts)
    return p if p.is_valid else None


def evaluate_image(gt_items, det_polys, iou_thr=IOU_THR, ign_thr=IGNORE_AREA_PRECISION_THR):
    gt_pols, gt_ign, det_pols, det_ign = [], [], [], []
    n_invalid_gt = n_invalid_det = 0
    for g in gt_items:
        p = _valid_poly(g['points'])
        if p is None:
            n_invalid_gt += 1
            continue
        gt_pols.append(p)
        if g['ignore']:
            gt_ign.append(len(gt_pols) - 1)
    for d in det_polys:
        p = _valid_poly(d)
        if p is None:
            n_invalid_det += 1
            continue
        det_pols.append(p)
        for gi in gt_ign:
            if p.area > 0 and gt_pols[gi].intersection(p).area / p.area > ign_thr:
                det_ign.append(len(det_pols) - 1)
                break
    gt_ign_s, det_ign_s = set(gt_ign), set(det_ign)
    gt_hit = np.zeros(len(gt_pols), dtype=bool)
    det_hit = np.zeros(len(det_pols), dtype=bool)
    pairs = []
    if gt_pols and det_pols:
        iou = np.zeros((len(gt_pols), len(det_pols)))
        for i, g in enumerate(gt_pols):
            for j, d in enumerate(det_pols):
                inter = g.intersection(d).area
                if inter > 0:
                    iou[i, j] = inter / g.union(d).area
        for i in range(len(gt_pols)):
            for j in range(len(det_pols)):
                if (not gt_hit[i] and not det_hit[j] and i not in gt_ign_s and j not in det_ign_s
                        and iou[i, j] > iou_thr):
                    gt_hit[i] = det_hit[j] = True
                    pairs.append((i, j, float(iou[i, j])))
    return {
        'n_gt_care': len(gt_pols) - len(gt_ign), 'n_det_care': len(det_pols) - len(det_ign),
        'matched': len(pairs), 'n_gt_ignore': len(gt_ign), 'n_det_ignore': len(det_ign),
        'n_invalid_gt': n_invalid_gt, 'n_invalid_det': n_invalid_det, 'pairs': pairs,
        'gt_pols': gt_pols, 'det_pols': det_pols, 'gt_ign': gt_ign_s, 'det_ign': det_ign_s,
        'gt_hit': gt_hit, 'det_hit': det_hit,
    }


def combine(results):
    m = sum(r['matched'] for r in results)
    g = sum(r['n_gt_care'] for r in results)
    d = sum(r['n_det_care'] for r in results)
    P = m / d if d else 0.0
    R = m / g if g else 0.0
    H = 2 * P * R / (P + R) if P + R else 0.0
    return {'precision': P, 'recall': R, 'hmean': H, 'matched': m, 'gt_care': g, 'det_care': d,
            'gt_ignore': sum(r['n_gt_ignore'] for r in results),
            'det_ignore': sum(r['n_det_ignore'] for r in results),
            'invalid_gt': sum(r['n_invalid_gt'] for r in results),
            'invalid_det': sum(r['n_invalid_det'] for r in results)}

In [ ]:
IMG_DIR, GT_SRC, GT_FMT = resolve_icdar15([INPUT_ROOT])
rrc = {s['image_id']: s for s in load_icdar15(IMG_DIR, GT_SRC, GT_FMT)}
official = parse_paddle_label(LABEL)
print('ảnh:', IMG_DIR, '| GT gốc:', GT_SRC, '| nhãn chính thức:', LABEL)


def canon_items(items):
    return sorted((tuple(np.asarray(g['points'], dtype=np.float64).round(3).reshape(-1).tolist()), g['text']) for g in items)


mismatch = [f'img_{k}' for k in sorted(official) if canon_items(official[k]) != canon_items(rrc[f'img_{k}']['gt'])]
n_box = sum(len(v) for v in official.values())
n_ign = sum(g['ignore'] for v in official.values() for g in v)
LABEL_AUDIT = {'official_images': len(official), 'rrc_images': len(rrc), 'same_image_ids': {f'img_{k}' for k in official} == set(rrc),
               'official_boxes': n_box, 'official_ignore': n_ign, 'official_care': n_box - n_ign,
               'images_with_label_mismatch_vs_rrc': mismatch}
write_json(ART / 'label_audit.json', LABEL_AUDIT)
print(json.dumps({k: (v if k != 'images_with_label_mismatch_vs_rrc' else f'{len(v)} ảnh {v[:5]}') for k, v in LABEL_AUDIT.items()}, indent=2))
assert LABEL_AUDIT['same_image_ids'] and len(official) == EXPECTED_N_IMAGES

# Bố trí thư mục đúng như config mong đợi: <data_dir>/ch4_test_images/img_*.jpg
DATA_DIR.mkdir(parents=True, exist_ok=True)
link = DATA_DIR / 'ch4_test_images'
if not link.exists():
    link.symlink_to(IMG_DIR, target_is_directory=True)
missing = [ln.split('\t', 1)[0] for ln in LABEL.read_text(encoding='utf-8').splitlines()
           if ln.strip() and not (DATA_DIR / ln.split('\t', 1)[0]).is_file()]
assert not missing, f'{len(missing)} ảnh trong file nhãn không tìm thấy: {missing[:5]}'
print('đủ 500 ảnh cho file nhãn chính thức trong', DATA_DIR)

## 4. Chạy `tools/eval.py` chính thức

Chỉ override **đường dẫn** (weights, data_dir, file nhãn) — không đổi tham số mô hình, tiền/hậu xử lý hay thước đo. Toàn bộ output lưu ở `official_eval.log`.

In [ ]:
EVAL_CMD = [sys.executable, 'tools/eval.py', '-c', CONFIG_REL, '-o',
            f'Global.checkpoints={WEIGHTS}',
            f'Eval.dataset.data_dir={DATA_DIR}/',
            f'Eval.dataset.label_file_list=[{LABEL}]']
env_eval = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
t0 = time.perf_counter()
p = subprocess.run(EVAL_CMD, capture_output=True, text=True, cwd=REPO, env=env_eval)
EVAL_SECONDS = time.perf_counter() - t0
log_text = f'$ cd {REPO} && CUDA_VISIBLE_DEVICES=0 {" ".join(EVAL_CMD)}\n\n' + p.stdout + p.stderr
(ART / 'official_eval.log').write_text(log_text, encoding='utf-8')
print('\n'.join(log_text.splitlines()[-25:]))
if p.returncode != 0:
    raise RuntimeError(f'tools/eval.py lỗi (exit {p.returncode}) — xem official_eval.log')

tail = log_text[log_text.rfind('metric eval ***************'):]
OFFICIAL_METRIC = {k: float(v) for k, v in re.findall(r'\b(precision|recall|hmean|fps):\s*([-+0-9.eE]+)', tail)}
assert {'precision', 'recall', 'hmean'} <= set(OFFICIAL_METRIC), f'không đọc được metric từ log: {tail[:500]}'
assert 'resume from' in log_text and str(WEIGHTS.with_suffix('')) in log_text, 'log không cho thấy đã nạp đúng weights'
# load_model chỉ CẢNH BÁO rồi chạy tiếp khi tên/shape tham số lệch -> model một phần ngẫu nhiên mà vẫn ra số. Coi là lỗi.
WEIGHTS_LOAD_CHECK = {'missing_keys_warnings': log_text.count('not in loaded params'),
                      'shape_mismatch_warnings': log_text.count('not matched with loaded params shape')}
print('kiểm tra nạp weights:', WEIGHTS_LOAD_CHECK)
assert not any(WEIGHTS_LOAD_CHECK.values()), 'weights KHÔNG khớp kiến trúc model — kết quả không dùng được, xem official_eval.log'
print(f'\nOFFICIAL  P={OFFICIAL_METRIC["precision"]:.4f}  R={OFFICIAL_METRIC["recall"]:.4f}  Hmean={OFFICIAL_METRIC["hmean"]:.4f}  ({EVAL_SECONDS:.0f}s)')

## 5. Kết quả + so sánh với số đã công bố

In [ ]:
H = OFFICIAL_METRIC['hmean']
METRICS = {
    'model': 'PP-OCRv5_server_det',
    'evaluation_type': 'official_pipeline_on_public_benchmark',
    'pipeline': {'script': 'tools/eval.py', 'repo_tag': PADDLEOCR_TAG, 'repo_commit': REPO_COMMIT, 'config': CONFIG_REL,
                 'overrides': EVAL_CMD[EVAL_CMD.index('-o') + 1:], 'weights': downloads['weights'], 'label': downloads['label']},
    'benchmark': {'name': 'ICDAR2015 test', 'n_images': len(official), 'care_boxes': LABEL_AUDIT['official_care'],
                  'image_dir': str(IMG_DIR)},
    'official_metric': OFFICIAL_METRIC,
    'weights_load_check': WEIGHTS_LOAD_CHECK,
    'eval_seconds': EVAL_SECONDS,
    'published_reference': PUBLISHED,
    'previous_module_run': PREV_MODULE_RUN,
    'delta_vs_published_83_8_pp': round(100 * (H - 0.838), 2),
    'delta_vs_previous_module_run_pp': round(100 * (H - PREV_MODULE_RUN['hmean']), 2),
    'label_audit': LABEL_AUDIT,
    'env': {k: env[k] for k in ('paddlepaddle', 'paddle_cuda', 'gpu', 'paddleocr_repo_commit')},
    'utc_time': RUN_STARTED,
}
write_json(ART / 'metrics.json', METRICS)

rows = [
    ('Kết quả này: tools/eval.py chính thức', 'ICDAR2015 test (500)', 'công khai',
     f'{100 * OFFICIAL_METRIC["precision"]:.2f}', f'{100 * OFFICIAL_METRIC["recall"]:.2f}', f'{100 * H:.2f}'),
    ('Lần trước: paddleocr.TextDetection (resize_long 960)', 'ICDAR2015 test (500)', 'công khai',
     f'{100 * PREV_MODULE_RUN["precision"]:.2f}', f'{100 * PREV_MODULE_RUN["recall"]:.2f}', f'{100 * PREV_MODULE_RUN["hmean"]:.2f}'),
    ('Công bố: docs Text Detection', 'general evaluation set', 'không', '—', '—', '83.80'),
    ('Công bố: docs PP-OCRv5 (TB 12 nhóm)', 'internal multi-scenario', 'không', '—', '—', '82.70'),
    ('Công bố: docs PP-OCRv5 (General Scenario)', 'internal multi-scenario', 'không', '—', '—', '79.70'),
]
table = '| Nguồn | Tập đánh giá | Công khai | P | R | Hmean |\n|---|---|---|---:|---:|---:|\n' + '\n'.join('| ' + ' | '.join(r) + ' |' for r in rows)
print(table)
print(f'\nChênh với 83,8 công bố: {METRICS["delta_vs_published_83_8_pp"]:+.2f} điểm % (KHÁC tập dữ liệu — không phải sai số reproduce)')
print(f'Chênh với lần chạy module trước: {METRICS["delta_vs_previous_module_run_pp"]:+.2f} điểm % (do resize 736-min vs 960-long và weights training vs inference)')

In [ ]:
availability_md = f"""# PP-OCRv5_server_det — số công bố và khả năng reproduce

## Số công bố
- Docs Text Detection: Hmean **83,8** trên "general evaluation set" — không mô tả số ảnh/nguồn, **không công khai**.
  ({PUBLISHED['docs_text_detection_module']['url']})
- Docs PP-OCRv5: bảng 12 nhóm trên "internal complex evaluation sets", trung bình **0,827**, General Scenario 0,797 — **không công khai**.
  ({PUBLISHED['docs_ppocrv5_intro_avg']['url']})
- Tech report PaddleOCR 3.0 (arXiv 2507.05595) và PP-OCRv5 (arXiv 2603.24373): **không có bảng detection nào**.
- Bộ 2.677 ảnh trên trang docs là bộ **đo tốc độ**, không gắn với Hmean 83,8 (handoff ghi nhầm).

## Trạng thái
- Exact reproduction 83,8: **NOT VERIFIED** — không có tập dữ liệu gốc.
- Đã làm giống hệt bản chính thức: code đánh giá `tools/eval.py` (tag {PADDLEOCR_TAG}, commit {REPO_COMMIT[:12]}), config
  `{CONFIG_REL}` không sửa, weights `PP-OCRv5_server_det_pretrained.pdparams` (sha256 {downloads['weights']['sha256'][:16]}…),
  nhãn `test_icdar2015_label.txt` chính thức, thước đo `DetMetric`.
- Kết quả trên ICDAR2015 test (tập Eval mặc định trong config chính thức):
  **P {100 * OFFICIAL_METRIC['precision']:.2f} · R {100 * OFFICIAL_METRIC['recall']:.2f} · Hmean {100 * H:.2f}**.

## Đã kiểm tra (ngày 2026-10-31)
- [x] Trang docs Text Detection — không có link tải tập đánh giá.
- [x] Trang docs PP-OCRv5 — chỉ ghi "internal", không có link.
- [x] arXiv 2507.05595 — không có số detection.
- [x] arXiv 2603.24373 — không có số detection.
- [ ] GitHub issues/discussions PaddleOCR (điền nếu đã hỏi)
"""
(ART / 'official_benchmark_availability.md').write_text(availability_md, encoding='utf-8')
(ART / 'evaluation_type.txt').write_text(
    'official_pipeline_on_public_benchmark\n'
    'PaddleOCR tools/eval.py + official config + official weights + official ICDAR2015 test labels.\n'
    'NOT an exact reproduction of the published Hmean 83.8 (its evaluation set is internal and unpublished).\n',
    encoding='utf-8')
print(availability_md)

listing = [f'{sha256_file(f)}  {f.relative_to(ART.parent)}' for f in sorted(ART.rglob('*'))
           if f.is_file() and f.name != 'artifacts_sha256.txt']
(ART / 'artifacts_sha256.txt').write_text('\n'.join(listing) + '\n', encoding='utf-8')
zip_path = shutil.make_archive(str(WORK / 'phase1_paddle_official'), 'zip', root_dir=ART.parent.parent, base_dir='artifacts')
print('zip ->', zip_path, '| sha256', sha256_file(zip_path))

## Gate 1 — phần Paddle
- [ ] `official_eval.log` có dòng `resume from …PP-OCRv5_server_det_pretrained` và khối `metric eval`.
- [ ] `label_audit.json`: nhãn chính thức trùng GT gốc ICDAR2015 (0 ảnh lệch).
- [ ] `git_commit.txt`: repo đúng tag v3.3.3, không file nào bị sửa.
- [ ] Bảng so sánh: ghi rõ 83,8 là **khác tập dữ liệu**, trạng thái **NOT VERIFIED**.
- [ ] Nhóm đồng ý bằng văn bản đây là mức "giống paper" tối đa làm được → sang Phase 2.